# Exercise 1: CT organ classifier from a NumPy file

Same task as `ct3_buggy.ipynb` (bladder / heart / liver), but the data is read from the `.npz` file through a hand-written `Dataset`, the model uses BatchNorm, and training keeps the best checkpoint.

**Interviewer:** "This notebook doesn't work, and once it runs the results look strange. Walk me through it, tell me what you're checking, and fix it."

There are **9 problems**: 3 crash the code, 6 are silent (the code runs, but the result is wrong).

**How to work** (same as the real interview)
1. Read the whole notebook first and say out loud what each part does.
2. Run the cells from the top and fix the crashes one at a time.
3. Once it runs, go through it again for the silent bugs: check shapes, dtypes, value ranges, split sizes, train vs eval mode, and whether the numbers make sense.

*Level: Warm-up, about 30 minutes. In Colab: Runtime → Change runtime type → T4 GPU (it also runs on a CPU, just slower).*

In [ ]:
import copy
import os

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

SEED = 0
BATCH_SIZE = 64
EPOCHS = 4
LR = 1e-3
KEEP = [0, 3, 6]                       # original OrganAMNIST labels we keep
NAMES = ["bladder", "heart", "liver"]  # their new labels: 0, 1, 2

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


def find_data_dir():
    """Finds organamnist_64.npz near this notebook, or downloads it (for example on Colab)."""
    for d in ["data", "../data", "../../data", "/content/data"]:
        if os.path.exists(os.path.join(d, "organamnist_64.npz")):
            return d
    import subprocess
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "medmnist"], check=True)
    import medmnist
    os.makedirs("data", exist_ok=True)
    medmnist.OrganAMNIST(split="train", download=True, size=64, root="data")
    return "data"

### Data

In [ ]:
data = np.load(os.path.join(find_data_dir(), "organamnist_64.npz"))


def select(split):
    imgs = data[f"{split}_images"]                  # (N, 64, 64) uint8
    labels = data[f"{split}_labels"].reshape(-1)    # (N, 1) -> (N,)
    mask = np.isin(labels, KEEP)
    new_labels = labels[mask]
    return imgs[mask], new_labels


x_train, y_train = select("train")
x_val, y_val = select("val")
x_test, y_test = select("val")
for name, y in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{name:5s} n={len(y):5d}  per class: {np.bincount(y, minlength=3)}")

pixels = np.concatenate([x_train, x_val, x_test])
MEAN = float(pixels.mean() / 255.0)
STD = float(pixels.std() / 255.0)
print(f"mean={MEAN:.3f} std={STD:.3f}")

In [ ]:
class CTDataset(Dataset):
    def __init__(self, imgs, labels, augment=False):
        self.imgs = imgs
        self.labels = torch.from_numpy(labels).long()
        self.augment = augment

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, i):
        x = torch.from_numpy(self.imgs[i]).float()
        x = (x - MEAN) / STD
        if self.augment:
            dy, dx = torch.randint(-3, 4, (2,)).tolist()
            x = torch.roll(x, shifts=(dy, dx), dims=(1, 2))
        return x, self.labels[i]


train_ds = CTDataset(x_train, y_train, augment=True)
val_ds = CTDataset(x_val, y_val)
test_ds = CTDataset(x_test, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256)
test_loader = DataLoader(test_ds, batch_size=256)

# sanity check: always look at one batch before training
x, y = next(iter(train_loader))
print("batch:", tuple(x.shape), x.dtype, f"mean={x.mean():.2f} std={x.std():.2f}", "labels:", y[:10].tolist())

### Model

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),   # -> 16 x 32 x 32
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),  # -> 32 x 16 x 16
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),  # -> 64 x 8 x 8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(64 * 16 * 16, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))   # raw logits; CrossEntropyLoss applies log-softmax itself


model = SmallCNN(num_classes=len(NAMES)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

### Training and evaluation

In [ ]:
def train_one_epoch(model, loader):
    total_loss, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss * x.size(0)
        n += x.size(0)
    return total_loss / n


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        total_loss += criterion(logits, y).item() * x.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += x.size(0)
    return total_loss / n, correct / n


best_acc, best_state = 0.0, None
for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)
    print(f"epoch {epoch}: train loss {train_loss:.3f} | val loss {val_loss:.3f} acc {val_acc:.3f}")
    if val_acc > best_acc:
        best_acc = val_acc
        best_state = model.state_dict()

In [ ]:
model.load_state_dict(best_state)
_, check_acc = evaluate(model, val_loader)
print(f"best val acc {best_acc:.3f}, re-checked after loading: {check_acc:.3f}")   # these two numbers must match
test_loss, test_acc = evaluate(model, test_loader)
print(f"TEST acc {test_acc:.3f} on {len(test_ds)} images (looked at once, after choosing the model on validation)")

### Your notes
Write each bug like this:
> Bug: cell … → why it's wrong → my fix → crash or silent?